# X-means Clustering

**Authors:** Abderrahmane Salmi, Ricardo Talarico, Lorenzo Allegrini

## Imports

In [1]:
import numpy as np
import pandas as pd
import math
import seaborn as sns
from datetime import datetime
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score

from pyclustering.cluster.xmeans import xmeans
from pyclustering.cluster.center_initializer import kmeans_plusplus_initializer
from pyclustering.utils import read_sample

In [2]:
tracks_df = pd.read_csv("../datasets/tracks_cleaned.csv")

In [3]:
# Print all feature names
tracks_original_features = tracks_df.columns.tolist()
print(tracks_original_features)

# Print total number of features
print(f"\nTotal number of features: {len(tracks_original_features)}")

['swear_IT', 'swear_EN', 'year', 'n_tokens', 'tokens_per_sent', 'char_per_tok', 'lexical_density', 'avg_token_per_clause', 'bpm', 'centroid', 'rolloff', 'flux', 'flatness', 'spectral_complexity', 'pitch', 'loudness', 'album_type', 'explicit', 'popularity', 'duration_sec', 'swear_ratio', 'aggressiveness', 'relative_popularity', 'release_season', 'has_collaboration', 'song_age']

Total number of features: 26


## Features Selection

We tried to select features that capture different aspects, like: linguistic, audio, popularity, etc.

In [4]:
selected_features = [
    # linguistic / lyrics
    # 'n_tokens',
    # 'swear_IT',
    # 'lexical_density',

    # audio features
    # 'bpm',
    'centroid',
    'loudness',
    'spectral_complexity',
    # 'flux',
    'rolloff',

    # popularity
    'popularity',
    # 'relative_popularity',

    # time-related
    # 'season_autumn', 'season_spring', 'season_summer', 'season_winter',

    # engineered features
    # 'has_collaboration',
    'aggressiveness',
    # 'song_age',
    # 'duration_sec'
]

In [5]:
#Extract feature matrix
X = tracks_df[selected_features].copy()

## Preprocessing

In [6]:
# Check for NaN values
print(X.isna().sum())

centroid               0
loudness               0
spectral_complexity    0
rolloff                0
popularity             0
aggressiveness         0
dtype: int64


**Normalization**: We tried to normalize using both z-score and min-max, to see which one works best for different algorithms.

In [7]:
# Scale the features (z-score)
from sklearn.preprocessing import StandardScaler

std_scaler = StandardScaler()
X_zscore = std_scaler.fit_transform(X)

In [8]:
# Scale the features (Min-Max)
from sklearn.preprocessing import MinMaxScaler

minmax_scaler = MinMaxScaler()
X_minmax = minmax_scaler.fit_transform(X)

## X-Means

In [9]:
X_scaled = X_minmax

In [13]:
# FIX FOR THE ATTRIBUTE ERROR
import numpy as np
import warnings
np.warnings = warnings # This fixes the compatibility issue with pyclustering

from pyclustering.cluster.xmeans import xmeans
from pyclustering.cluster.center_initializer import kmeans_plusplus_initializer
from sklearn.metrics import silhouette_score
import pandas as pd

# 1. Prepare Data (Ensuring it is a numpy array for pyclustering)
# Use the same X_scaled you used for K-Means
X_input = X_scaled.tolist() if isinstance(X_scaled, np.ndarray) else X_scaled.values.tolist()

# 2. Initialize Centers (Starting with 2 clusters)
# This is where your previous error happened
initial_centers = kmeans_plusplus_initializer(X_input, 2).initialize()

# 3. Run X-Means
# kmax is the maximum clusters it will try to split into
xmeans_instance = xmeans(X_input, initial_centers, kmax=20)
xmeans_instance.process()

# 4. Extract Results
clusters = xmeans_instance.get_clusters()
centers = xmeans_instance.get_centers()
n_clusters_found = len(clusters)

print(f"X-Means successfully found the optimal number of clusters: {n_clusters_found}")

# 5. Convert results to standard labels
labels_xmeans = np.zeros(len(X_input), dtype=int)
for cluster_id, indices in enumerate(clusters):
    for index in indices:
        labels_xmeans[index] = cluster_id

# 6. Evaluation
score = silhouette_score(X_scaled, labels_xmeans)
print(f"Silhouette Score for X-Means: {score:.4f}")

# 7. Add to DataFrame for characterization
tracks_df['xmeans_cluster'] = labels_xmeans

X-Means successfully found the optimal number of clusters: 2
Silhouette Score for X-Means: 0.2354


In [14]:
# Group by the new X-Means clusters and look at the means
xmeans_profiles = tracks_df.groupby('xmeans_cluster')[selected_features].mean()

# Calculate deviation from the global mean to see what makes each cluster special
global_mean = tracks_df[selected_features].mean()
xmeans_deviation = xmeans_profiles - global_mean

print("X-Means Cluster Profiles (Deviation from Average):")
display(xmeans_deviation)

X-Means Cluster Profiles (Deviation from Average):


,centroid,loudness,spectral_complexity,rolloff,popularity,aggressiveness
xmeans_cluster,,,,,,
0,-0.016158,-2.045089,-3.961215,-0.292653,1.717735,-0.292656
1,0.019651,2.487198,4.817556,0.355919,-2.089078,0.355923
